In [1]:
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load data
train_df = pd.read_pickle("../../data/processed/features_train.pkl")
val_df = pd.read_pickle("../../data/processed/features_val.pkl")
test_df = pd.read_pickle("../../data/processed/features_test.pkl")

In [2]:
# 2. Check number of used features
metadata_cols = ['user_id', 'set', 'label', 'weight', 'height', 'age', 'gender']
feature_cols = [col for col in train_df.columns if col not in metadata_cols]

print(f"Number of features used for training: {len(feature_cols)}")

# 3. Split features and labels
X_train = train_df[feature_cols]
y_train = train_df["label"]

X_val = val_df[feature_cols]
y_val = val_df["label"]

X_test = test_df[feature_cols]
y_test = test_df["label"]

# 4. Define model
model = RandomForestClassifier( 
    n_estimators=300, # Số cây tối đa
    random_state=42, # Bảo đảm tất cả các lần chạy đều ra 1 KQ
    max_features = "sqrt", # Mỗi node thường lấy từng này features đã train
    class_weight="balanced", # Vì sự imbalance của các window ta đã thấy ở FE
)

# 5. Train
model.fit(X_train, y_train)

# 6. Evaluation
y_val_pred = model.predict(X_val)
print("\nValidation accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

y_test_pred = model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))

Number of features used for training: 30

Validation accuracy: 0.855012853470437
              precision    recall  f1-score   support

           0       0.83      0.76      0.80       184
           1       0.53      0.80      0.64       229
           2       0.89      0.67      0.77       467
           3       1.00      0.79      0.88       189
           4       1.00      1.00      1.00       413
           5       0.89      1.00      0.94       463

    accuracy                           0.86      1945
   macro avg       0.86      0.84      0.84      1945
weighted avg       0.88      0.86      0.86      1945

Test accuracy: 0.8967459324155194
              precision    recall  f1-score   support

           0       0.77      0.81      0.79       139
           1       0.65      0.88      0.75       184
           2       0.95      0.72      0.82       398
           3       0.83      0.97      0.90       156
           4       1.00      1.00      1.00       349
           5     

Random Forest là một phương pháp ensemble learning, trong đó nhiều Decision Tree được huấn luyện song song trên các tập dữ liệu khác nhau.
Kết quả dự đoán cuối cùng được xác định bằng bỏ phiếu đa số (majority voting) - nhãn xuất hiện nhiều nhất sẽ là kết quả cuối cùng

- Support: Số lượng mẫu thực tế của mỗi lớp trong tập dữ liệu.
- Recall: Tỷ lệ các mẫu thuộc lớp đó được mô hình dự đoán đúng (khả năng không bỏ sót mẫu).
- Precision: Tỷ lệ dự đoán đúng trong số các mẫu được mô hình dự đoán là thuộc lớp đó (mức độ chính xác của dự đoán).
- F1-score: Trung bình điều hòa giữa Precision và Recall, phản ánh mức độ cân bằng giữa độ chính xác và khả năng phát hiện.
- Macro average: Trung bình các chỉ số trên tất cả các lớp với trọng số như nhau, dùng để đánh giá mô hình có bị học lệch giữa các lớp hay không.
- Weighted average: Trung bình các chỉ số có tính đến số lượng mẫu (support) của từng lớp, phản ánh hiệu năng thực tế của mô hình trên toàn bộ dữ liệu.

- Validation accuracy ≈ 85.5%
+ Lớp 4 và 5 có kết quả rất cao, lớp 3 thấp hơn chút
+ Lớp 0, 1 và 2 thấp hơn cả, chứng tỏ model vẫn còn nhầm lẫn với lớp khác

- Test accuracy ≈ 89.7%
+ Lớp 4, 5 được dự đoán chính xác tuyệt đối, lớp 2 và 3 tương đối chính xác(trong đó lớp 3 với recall khá cao - hầu hết các mẫu đều nhận diện đúng)
+ Lớp 0 và 1 kém chính xác nhất

In [3]:
# Test with a single label
LABEL_NAME = {
    0: "Downstairs",
    1: "Jogging",
    2: "Sitting",
    3: "Standing",
    4: "Upstairs",
    5: "Walking"
}

sample = 10  # chọn 1 sample bất kỳ

x_demo = X_test.iloc[sample:sample+1]
y_true = y_test.iloc[sample]

y_pred = model.predict(x_demo)[0]

print("Try with a random label")
print("Prediction:", LABEL_NAME[y_pred])
print("True label:", LABEL_NAME[y_true])

Try with a random label
Prediction: Downstairs
True label: Downstairs
